In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm, kstest

# ============================================================
# Original load data — 13 timesteps x 4 nodes (buses 1-4)
# Node 5 is the slack bus (no load data needed)
# ============================================================
data_real = np.array([
    [0.332, 0.064, 0.084, 0.12],
    [0.236, 0.164, 0.276, 0.064],
    [0.224, 0.708, 1.572, 0.072],
    [0.36,  3.44,  1.188, 0.18],
    [1.332, 2.176, 0.484, 1.464],
    [1.516, 3.02,  0.316, 0.624],
    [0.92,  0.916, 0.404, 2.772],
    [0.752, 0.64,  0.396, 1.464],
    [1.828, 0.684, 0.576, 0.576],
    [3.568, 0.564, 0.828, 0.428],
    [0.78,  0.356, 0.728, 0.348],
    [0.856, 0.22,  0.308, 0.12],
    [0.684, 0.528, 0.256, 0.44],
])

# ============================================================
# Step 1 — Fit a log-normal distribution to each node
# Log-normal is physically motivated for power loads:
#   - always positive
#   - right-skewed (occasional high demand spikes)
#   - well-established in load modelling literature
# ============================================================
N_SYNTHETIC = 500   # number of synthetic timesteps to generate
np.random.seed(42)  # reproducibility

n_nodes = data_real.shape[1]
log_means = np.zeros(n_nodes)
log_stds  = np.zeros(n_nodes)

print("=== Log-normal fit parameters per node ===")
for i in range(n_nodes):
    log_data      = np.log(data_real[:, i])
    log_means[i]  = np.mean(log_data)
    log_stds[i]   = np.std(log_data)
    print(f"  Node {i+1}: μ_log = {log_means[i]:.3f},  σ_log = {log_stds[i]:.3f}")

# ============================================================
# Step 2 — Generate synthetic samples independently per node
# Nodes are nearly uncorrelated (checked in analysis),
# so independent sampling is valid here.
# ============================================================
data_synthetic = np.zeros((N_SYNTHETIC, n_nodes))
for i in range(n_nodes):
    data_synthetic[:, i] = np.random.lognormal(
        mean=log_means[i],
        sigma=log_stds[i],
        size=N_SYNTHETIC
    )

print(f"\n=== Synthetic dataset: {N_SYNTHETIC} timesteps generated ===")

# ============================================================
# Step 3 — Validate: compare real vs synthetic statistics
# ============================================================
print("\n=== Validation: real vs synthetic statistics ===")
print(f"{'Node':<8}{'Real mean':<14}{'Syn mean':<14}{'Real std':<14}{'Syn std':<14}{'KS p-value'}")
for i in range(n_nodes):
    real_mean = np.mean(data_real[:, i])
    syn_mean  = np.mean(data_synthetic[:, i])
    real_std  = np.std(data_real[:, i])
    syn_std   = np.std(data_synthetic[:, i])

    # Kolmogorov-Smirnov test: does synthetic match the fitted log-normal?
    s, loc, scale = log_stds[i], 0, np.exp(log_means[i])
    ks_stat, ks_p = kstest(data_synthetic[:, i], 'lognorm', args=(s, loc, scale))

    print(f"  Node {i+1}   {real_mean:<14.3f}{syn_mean:<14.3f}{real_std:<14.3f}{syn_std:<14.3f}{ks_p:.3f}")

# ============================================================
# Step 4 — Plot: real data vs synthetic distribution
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Real vs Synthetic Load Data — Per Node", fontsize=14, fontweight='bold')

colors_real = '#2c7bb6'
colors_syn  = '#d7191c'

for i, ax in enumerate(axes.flat):
    # Synthetic histogram
    ax.hist(data_synthetic[:, i], bins=40, density=True,
            alpha=0.5, color=colors_syn, label='Synthetic (log-normal)')

    # Overlay fitted log-normal PDF
    x = np.linspace(0.001, data_synthetic[:, i].max() * 1.1, 300)
    pdf = lognorm.pdf(x, s=log_stds[i], scale=np.exp(log_means[i]))
    ax.plot(x, pdf, color=colors_syn, linewidth=2, label='Fitted PDF')

    # Real data as vertical lines
    for val in data_real[:, i]:
        ax.axvline(val, color=colors_real, alpha=0.6, linewidth=1.2)
    ax.axvline(data_real[0, i], color=colors_real, alpha=0.6,
               linewidth=1.2, label='Real data points')

    ax.set_title(f'Node {i+1}', fontweight='bold')
    ax.set_xlabel('Load (p.u.)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('images/load_distribution_validation.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nPlot saved.")

# ============================================================
# Step 5 — Save synthetic dataset as .npy for use in project
# ============================================================
np.save('data/load_synthetic.npy', data_synthetic)
np.save('data/load_real.npy', data_real)
print("Datasets saved: load_synthetic.npy, load_real.npy")

# ============================================================
# Usage note
# ============================================================
print("""
=== How to use in your project ===

  import numpy as np
  data_synthetic = np.load('load_synthetic.npy')  # shape: (500, 4)
  data_real      = np.load('load_real.npy')        # shape: (13, 4)

  # data_synthetic[:,0] → Node 1 loads, 500 timesteps
  # data_synthetic[:,1] → Node 2 loads, 500 timesteps
  # etc.

  # Use data_synthetic as your normal operation baseline
  # Use data_real as a validation check
""")

=== Log-normal fit parameters per node ===
  Node 1: μ_log = -0.280,  σ_log = 0.790
  Node 2: μ_log = -0.497,  σ_log = 1.093
  Node 3: μ_log = -0.804,  σ_log = 0.726
  Node 4: μ_log = -1.023,  σ_log = 1.149

=== Synthetic dataset: 500 timesteps generated ===

=== Validation: real vs synthetic statistics ===
Node    Real mean     Syn mean      Real std      Syn std       KS p-value
  Node 1   1.030         1.046         0.874         1.130         0.955
  Node 2   1.037         1.115         1.065         1.477         0.747
  Node 3   0.570         0.631         0.402         0.499         0.093
  Node 4   0.667         0.707         0.758         1.170         0.578

Plot saved.
Datasets saved: load_synthetic.npy, load_real.npy

=== How to use in your project ===

  import numpy as np
  data_synthetic = np.load('load_synthetic.npy')  # shape: (500, 4)
  data_real      = np.load('load_real.npy')        # shape: (13, 4)

  # data_synthetic[:,0] → Node 1 loads, 500 timesteps
  # data_syn